# PISA Chain間 Interface 解析

PDB ID を入力として、RCSB から mmCIF を取得し、ローカル PISA により chain 間 interface を解析します。

**入力:** PDB ID（例: `6XEZ`）

**データ (`data/`):** `{pdbid}.cif` — RCSB から自動ダウンロード（既存なら再取得しない）

**出力 (`results/`):**
- `{PDBID}_chainA-E_pisa_interface.csv` — interface 残基情報（`chain`, `resnum` で RSCC / Q-score 等と merge 可能）
- `{PDBID}_chainA-E_BSA.png` — 残基番号 vs BSA 棒グラフ


## 0. Python 依存パッケージ

初回のみ実行してください。


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED = ["pandas", "numpy", "matplotlib", "seaborn", "biopython"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(
    "Bio" if pkg == "biopython" else pkg
) is None]

if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required Python packages are already installed.")



## 1. ライブラリ読み込み & PISA 実行ファイル確認


In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import uuid
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from Bio.PDB import MMCIFParser, PPBuilder
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

try:
    import gemmi
    HAS_GEMMI = True
except ImportError:
    HAS_GEMMI = False

RCSB_DOWNLOAD_URL = "https://files.rcsb.org/download"
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry"

PISA_CANDIDATES = [
    Path("/Applications/ccp4-8.0/bin/pisa"),
    Path("/Applications/ccp4-7.1/bin/pisa"),
    Path("/usr/local/bin/pisa"),
]


def find_pisa_executable() -> Path | None:
    env_path = os.environ.get("PISA_EXE")
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    which = shutil.which("pisa")
    if which:
        return Path(which)
    for candidate in PISA_CANDIDATES:
        if candidate.is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None


PISA_EXE = find_pisa_executable()
if PISA_EXE is None:
    raise FileNotFoundError(
        "PISA 実行ファイルが見つかりません。\n"
        "CCP4 をインストールし、pisa が PATH に含まれることを確認してください。\n"
        "例: /Applications/ccp4-8.0/bin/pisa\n"
        "環境変数 PISA_EXE にフルパスを設定することもできます。"
    )

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "results").exists() and (NOTEBOOK_DIR / "structure_tools" / "PISA").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "structure_tools" / "PISA"

DATA_DIR = NOTEBOOK_DIR / "data"
RESULTS_DIR = NOTEBOOK_DIR / "results"
PISA_WORK_DIR = NOTEBOOK_DIR / "work" / "pisa_sessions"
for d in (DATA_DIR, RESULTS_DIR, PISA_WORK_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"PISA executable : {PISA_EXE}")
print(f"gemmi available : {HAS_GEMMI}")
print(f"Data directory  : {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")



## 2. 入力パラメータ

PDB ID のみ指定します。


In [ ]:
# ===== ユーザー設定 =====
PDB_ID = "6XEZ"

PDB_ID = PDB_ID.strip().upper()
if len(PDB_ID) != 4:
    raise ValueError(f"PDB ID は4文字で指定してください: {PDB_ID!r}")

print(f"PDB ID: {PDB_ID}")



## 3. 関数定義


In [ ]:
from __future__ import annotations

def download_mmcif(pdb_id: str, data_dir: Path) -> Path:
    """RCSB から mmCIF をダウンロードする。既存ファイルは再取得しない。"""
    pdb_id = pdb_id.upper()
    cif_path = data_dir / f"{pdb_id.lower()}.cif"
    if cif_path.exists():
        print(f"Skip (already exists): {cif_path}")
        return cif_path

    url = f"{RCSB_DOWNLOAD_URL}/{pdb_id}.cif"
    print(f"Downloading: {url}")
    with urlopen(url, timeout=120) as resp:
        data = resp.read()
    cif_path.write_bytes(data)
    print(f"Saved: {cif_path} ({len(data):,} bytes)")
    return cif_path


def fetch_entry_metadata(pdb_id: str) -> dict:
    """RCSB REST API から entry メタデータを取得する。"""
    url = f"{RCSB_ENTRY_URL}/{pdb_id.upper()}"
    with urlopen(url, timeout=60) as resp:
        return json.loads(resp.read().decode())


def parse_structure(structure_path: Path) -> dict:
    """mmCIF からメタ情報を取得する。"""
    meta = {
        "pdb_id": structure_path.stem.upper()[:4],
        "title": None,
        "resolution": None,
        "method": None,
        "structure_path": structure_path,
    }

    cif_dict = MMCIF2Dict(str(structure_path))
    meta["pdb_id"] = cif_dict.get("_entry.id", [meta["pdb_id"]])[0].upper()
    meta["title"] = cif_dict.get("_struct.title", [None])[0]
    meta["method"] = cif_dict.get("_exptl.method", [None])[0]
    for key in ("_refine.ls_d_res_high", "_reflns.d_resolution_high", "_em_3d_reconstruction.resolution"):
        if key in cif_dict:
            try:
                meta["resolution"] = float(cif_dict[key][0])
                break
            except (ValueError, IndexError):
                pass
    return meta


def get_structure_metadata(pdb_id: str, structure_path: Path) -> dict:
    """ファイル解析 + RCSB API でメタ情報を補完する。"""
    meta = parse_structure(structure_path)
    meta["pdb_id"] = pdb_id.upper()
    try:
        entry = fetch_entry_metadata(pdb_id)
        if not meta["title"]:
            meta["title"] = entry.get("struct", {}).get("title")
        methods = [x.get("method", "") for x in entry.get("exptl", []) if x.get("method")]
        if methods:
            meta["method"] = "; ".join(methods)
        res = entry.get("rcsb_entry_info", {}).get("resolution_combined")
        if meta["resolution"] is None and res is not None:
            meta["resolution"] = float(res[0] if isinstance(res, list) else res)
    except Exception as exc:
        print(f"Warning: RCSB metadata fetch failed ({exc})")
    return meta


def _fetch_rcsb_chain_descriptions(pdb_id: str) -> dict[str, str]:
    """RCSB REST API から entity 説明を取得する。"""
    base = "https://data.rcsb.org/rest/v1/core"
    mapping: dict[str, str] = {}
    try:
        entry = fetch_entry_metadata(pdb_id)
        container = entry.get("rcsb_entry_container_identifiers", {})

        for entity_id in container.get("polymer_entity_ids", []):
            with urlopen(f"{base}/polymer_entity/{pdb_id.upper()}/{entity_id}", timeout=30) as resp:
                entity = json.loads(resp.read().decode())
            desc = entity.get("rcsb_polymer_entity", {}).get("pdbx_description", "Unknown")
            strands = entity.get("entity_poly", {}).get("pdbx_strand_id", "")
            for chain_id in [s.strip() for s in strands.replace(" ", "").split(",") if s.strip()]:
                mapping[chain_id] = desc

        for entity_id in container.get("non_polymer_entity_ids", []):
            with urlopen(f"{base}/nonpolymer_entity/{pdb_id.upper()}/{entity_id}", timeout=30) as resp:
                entity = json.loads(resp.read().decode())
            nonpoly = entity.get("pdbx_entity_nonpoly", {})
            desc = (
                nonpoly.get("name")
                or entity.get("rcsb_nonpolymer_entity", {}).get("pdbx_description")
                or nonpoly.get("comp_id")
                or "Non-polymer entity"
            )
            for chain_id in entity.get("rcsb_nonpolymer_entity_container_identifiers", {}).get("asym_ids", []):
                mapping[chain_id] = desc
    except Exception:
        return mapping
    return mapping


def get_chain_information(structure_path: Path, pdb_id: str) -> pd.DataFrame:
    """各 chain の molecule 名と残基数を表形式で返す。"""
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("model", str(structure_path))
    rcsb_names = _fetch_rcsb_chain_descriptions(pdb_id)

    rows = []
    for model in structure:
        for chain in model:
            chain_id = chain.id
            residues = [r for r in chain if r.id[0] == " "]
            molecule_name = rcsb_names.get(chain_id)
            if molecule_name is None:
                resnames = {r.get_resname() for r in residues[:20]}
                molecule_name = "/".join(sorted(resnames)) if resnames else "Unknown"
            rows.append({
                "chain_id": chain_id,
                "molecule_name": molecule_name,
                "residue_count": len(residues),
            })
        break
    return pd.DataFrame(rows)


def _normalize_pisa_chain(raw: str) -> str:
    """PISA 出力の chain 表記を正規化する（例: [ADP]A:1004 -> A）。"""
    raw = raw.strip()
    m = re.match(r"\[[^\]]+\]([A-Za-z0-9]+):\d+", raw)
    if m:
        return m.group(1)
    m = re.match(r"([A-Za-z0-9]+)", raw)
    return m.group(1) if m else raw


def _write_pisa_config(data_root: Path) -> Path:
    cfg_path = data_root / "pisa.cfg"
    if cfg_path.exists():
        return cfg_path
    result = subprocess.run([str(PISA_EXE), "-cfg-template"], capture_output=True, text=True, check=True)
    lines = result.stdout.splitlines()
    out = []
    for line in lines:
        if line.strip() == "DATA_ROOT":
            out.extend(["DATA_ROOT", str(data_root.resolve())])
            continue
        if out and out[-1] == str(data_root.resolve()):
            continue
        out.append(line)
    cfg_path.write_text("\n".join(out) + "\n")
    return cfg_path


def run_pisa(structure_path: Path, pdb_id: str, session_name: str | None = None, work_dir: Path | None = None) -> dict:
    """PISA -analyse を subprocess で実行する。"""
    work_dir = work_dir or PISA_WORK_DIR
    work_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = _write_pisa_config(work_dir)
    if session_name is None:
        session_name = f"pisa_{pdb_id.lower()}_{uuid.uuid4().hex[:8]}"
    cmd = [str(PISA_EXE), session_name, "-analyse", str(structure_path.resolve()), str(cfg_path)]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    stdout = proc.stdout or ""
    stderr = proc.stderr or ""
    if proc.returncode != 0:
        raise RuntimeError(f"PISA analyse failed (exit {proc.returncode}):\n{stdout}\n{stderr}")
    return {"session_name": session_name, "cfg_path": cfg_path, "work_dir": work_dir, "stdout": stdout}


def get_interface_summary(session_name: str, cfg_path: Path) -> pd.DataFrame:
    """PISA -list interfaces の出力を DataFrame に変換する。"""
    cmd = [str(PISA_EXE), session_name, "-list", "interfaces", str(cfg_path)]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    pattern = re.compile(r"^\s*(\d+)\s+\d+\s*\|\s*(\S+)\s*\|\s*(\S+)\s+.*?\|\s*([\d.]+)\s+([-\d.]+)")
    rows = []
    for line in proc.stdout.splitlines():
        m = pattern.match(line)
        if not m:
            continue
        rows.append({
            "ID": int(m.group(1)),
            "Chain1": _normalize_pisa_chain(m.group(2)),
            "Chain2": _normalize_pisa_chain(m.group(3)),
            "Area": float(m.group(4)),
            "deltaG": float(m.group(5)),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("PISA interface list が空です。")
    return df


def parse_chain_pair(chain_pair: str) -> tuple[str, str]:
    """'A-E' 形式の chain pair 文字列をパースする。"""
    s = chain_pair.strip().upper().replace(" ", "")
    for sep in ("-", ",", "/"):
        if sep in s:
            parts = [p.strip() for p in s.split(sep) if p.strip()]
            if len(parts) == 2:
                return parts[0], parts[1]
    raise ValueError(f"chain pair の形式が不正です: {chain_pair!r}（例: A-E）")


def _find_interface_serial(summary: pd.DataFrame, chain1: str, chain2: str) -> int:
    c1, c2 = chain1.strip().upper(), chain2.strip().upper()
    hit = summary[
        ((summary["Chain1"] == c1) & (summary["Chain2"] == c2))
        | ((summary["Chain1"] == c2) & (summary["Chain2"] == c1))
    ]
    if hit.empty:
        available = summary[["ID", "Chain1", "Chain2", "Area", "deltaG"]].to_string(index=False)
        raise ValueError(f"Interface {c1}-{c2} が見つかりません。\n{available}")
    return int(hit.iloc[0]["ID"])


def _parse_residue_section(lines: list[str], structure_label: str, interface_pair: str) -> list[dict]:
    residue_re = re.compile(
        r"^\s*\d+\s*\|\s*([Is])\s*\|\s*(\w+):(\w+)\s+(\d+)\s*\|\s*([HSDC\s])\s*\|\s*([\d.]+)\s+([\d.]+)\s+([-\d.]+)"
    )
    rows = []
    for line in lines:
        m = residue_re.match(line)
        if not m:
            continue
        rows.append({
            "interface_pair": interface_pair,
            "chain": m.group(2),
            "resname": m.group(3),
            "resnum": int(m.group(4)),
            "Structure1": structure_label,
            "HSDC": m.group(5).strip() or None,
            "ASA": float(m.group(6)),
            "BSA": float(m.group(7)),
            "delta_iG": float(m.group(8)),
        })
    return rows


def extract_interface_residues(
    session_name: str,
    cfg_path: Path,
    chain1: str,
    chain2: str,
    summary: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """指定 chain pair の interface 残基レポートを取得する。"""
    if summary is None:
        summary = get_interface_summary(session_name, cfg_path)
    serial = _find_interface_serial(summary, chain1, chain2)
    interface_pair = f"{chain1.upper()}-{chain2.upper()}"

    cmd = [str(PISA_EXE), session_name, "-detail", "interface", str(serial), str(cfg_path)]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    lines = proc.stdout.splitlines()

    rows = []
    section = None
    section_lines = []
    for line in lines:
        if "4. Interfacing Residues: Structure 1" in line:
            if section == "Structure 1" and section_lines:
                rows.extend(_parse_residue_section(section_lines, "Structure 1", interface_pair))
            section, section_lines = "Structure 1", []
            continue
        if "5. Interfacing Residues: Structure 2" in line:
            if section == "Structure 1" and section_lines:
                rows.extend(_parse_residue_section(section_lines, "Structure 1", interface_pair))
            section, section_lines = "Structure 2", []
            continue
        if section and (line.strip().startswith("Residues with most") or line.strip().startswith("----'")):
            rows.extend(_parse_residue_section(section_lines, section, interface_pair))
            section, section_lines = None, []
            continue
        if section:
            section_lines.append(line)
    if section and section_lines:
        rows.extend(_parse_residue_section(section_lines, section, interface_pair))

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"Interface {interface_pair} (ID={serial}) の残基情報を取得できませんでした。")
    return df.sort_values(["chain", "resnum"]).reset_index(drop=True)


def export_interface_csv(df: pd.DataFrame, pdb_id: str, chain1: str, chain2: str, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    c1, c2 = chain1.upper(), chain2.upper()
    out_path = out_dir / f"{pdb_id.upper()}_chain{c1}-{c2}_pisa_interface.csv"
    columns = ["interface_pair", "chain", "resname", "resnum", "Structure1", "HSDC", "ASA", "BSA", "delta_iG"]
    df[columns].to_csv(out_path, index=False)
    print(f"CSV saved: {out_path}")
    return out_path


def _chain_legend_label(chain_id: str, chain_df: pd.DataFrame | None) -> str:
    """凡例ラベル: Chain A - RNA-directed RNA polymerase"""
    label = f"Chain {chain_id}"
    if chain_df is not None and not chain_df.empty:
        hit = chain_df[chain_df["chain_id"] == chain_id]
        if not hit.empty:
            name = hit.iloc[0]["molecule_name"]
            if name and str(name) != "Unknown":
                label = f"Chain {chain_id} - {name}"
    return label


def plot_bsa_barplot(
    df: pd.DataFrame,
    pdb_id: str,
    chain1: str,
    chain2: str,
    out_dir: Path,
    chain_df: pd.DataFrame | None = None,
    dpi: int = 300,
) -> tuple[plt.Figure, Path]:
    """CSV データから Residue Number vs BSA の棒グラフを作成する。"""
    c1, c2 = chain1.upper(), chain2.upper()
    plot_df = df[df["BSA"] > 0].copy()
    if plot_df.empty:
        plot_df = df.copy()

    chain_colors = {c1: "#1f77b4", c2: "#d62728"}
    fig, ax = plt.subplots(figsize=(14, 5))
    for chain_id, sub in plot_df.groupby("chain"):
        color = chain_colors.get(chain_id, "#888888")
        label = _chain_legend_label(chain_id, chain_df)
        ax.bar(sub["resnum"], sub["BSA"], width=1.0, color=color, label=label, alpha=0.85)

    res_min = int(plot_df["resnum"].min())
    res_max = int(plot_df["resnum"].max())
    tick_start = (res_min // 50) * 50
    tick_end = ((res_max + 49) // 50) * 50
    ax.set_xticks(range(tick_start, tick_end + 1, 50))

    ax.set_xlabel("Residue Number")
    ax.set_ylabel("BSA (Å²)")
    ax.set_title(f"{pdb_id.upper()} Interface {c1}-{c2}")
    ax.legend()
    sns.despine(ax=ax)
    fig.tight_layout()

    out_dir.mkdir(parents=True, exist_ok=True)
    png_path = out_dir / f"{pdb_id.upper()}_chain{c1}-{c2}_BSA.png"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    print(f"Plot saved: {png_path}")
    return fig, png_path



## 4. 構造ダウンロード & エントリ概要

RCSB から mmCIF を取得し、PDB ID / Title / Resolution / Experimental Method と Chain 一覧を表示します。


In [ ]:
STRUCTURE_FILE = download_mmcif(PDB_ID, DATA_DIR)
meta = get_structure_metadata(PDB_ID, STRUCTURE_FILE)
chain_df = get_chain_information(STRUCTURE_FILE, PDB_ID)

print(f"PDB ID              : {meta['pdb_id']}")
print(f"Title               : {meta.get('title') or 'N/A'}")
if meta.get("resolution") is not None:
    print(f"Resolution          : {meta['resolution']:.2f} Å")
else:
    print("Resolution          : N/A")
print(f"Experimental Method : {meta.get('method') or 'N/A'}")
print(f"Structure file      : {STRUCTURE_FILE.resolve()}")

chain_df



## 5. PISA 実行 & Interface 一覧

PISA 実行後、検出された interface 一覧を DataFrame で表示します。


In [ ]:
pisa_info = run_pisa(STRUCTURE_FILE, PDB_ID)
interface_summary = get_interface_summary(pisa_info["session_name"], pisa_info["cfg_path"])
interface_summary = interface_summary.rename(columns={"deltaG": "ΔG"})
interface_summary


## 6. Interface 解析（A-E 固定）

chain pair **A-E** のみを自動解析します（ユーザー入力なし）。


In [ ]:
# A-E interface 固定（chain selection なし）
CHAIN1, CHAIN2 = "A", "E"
print(f"Selected interface: {CHAIN1} - {CHAIN2}")

selected_interface = interface_summary[
    ((interface_summary["Chain1"] == CHAIN1) & (interface_summary["Chain2"] == CHAIN2))
    | ((interface_summary["Chain1"] == CHAIN2) & (interface_summary["Chain2"] == CHAIN1))
]
if selected_interface.empty:
    raise ValueError(f"Interface {CHAIN1}-{CHAIN2} not found in PISA output.")
selected_interface


## 7. Interface 残基情報取得

選択した chain pair の interface residue report のみ解析します。

> **merge キー:** `interface_pair`, `chain`, `resnum` — 将来 RSCC / Q-score / Ramachandran と結合可能


In [ ]:
interface_residues = extract_interface_residues(
    pisa_info["session_name"],
    pisa_info["cfg_path"],
    CHAIN1,
    CHAIN2,
    interface_summary,
)
interface_residues.head(10)



## 8. CSV 出力

`results/{PDBID}_chainA-E_pisa_interface.csv` に保存します。


In [ ]:
csv_path = export_interface_csv(
    interface_residues,
    meta["pdb_id"],
    CHAIN1,
    CHAIN2,
    RESULTS_DIR,
)
csv_path



## 9. BSA 可視化 & 画像保存

CSV から Residue Number vs BSA の棒グラフを作成します（Chain1: 青, Chain2: 赤）。

保存先: `results/{PDBID}_chainA-E_BSA.png` (dpi=300)


In [ ]:
fig, png_path = plot_bsa_barplot(
    interface_residues,
    meta["pdb_id"],
    CHAIN1,
    CHAIN2,
    RESULTS_DIR,
    chain_df=chain_df,
    dpi=300,
)
plt.show()
png_path



## 10. 解析結果の確認

CSV を再読込し、`head()` と `describe()` で結果を確認します。


In [ ]:
result_csv = pd.read_csv(csv_path)

print("=== head() ===")
display(result_csv.head())

print("\n=== describe() ===")
display(result_csv.describe(include="all"))

print(f"\n出力ファイル:")
print(f"  CSV : {csv_path}")
print(f"  PNG : {png_path}")

